<div align="center">
    <span style="font-size:30px">
        <strong>
            <!-- Símbolo de Python -->
            <img
                src="https://cdn3.emoji.gg/emojis/1887_python.png"
                style="margin-bottom:-5px"
                width="30px" 
                height="30px"
            >
            <!-- Título -->
            Python para Geólogos
            <!-- Versión -->
            <img 
                src="https://img.shields.io/github/release/kevinalexandr19/manual-python-geologia.svg?style=flat&label=&color=blue"
                style="margin-bottom:-2px" 
                width="40px"
            >
        </strong>
    </span>
    <br>
    <span>
        <!-- Github del proyecto -->
        <a href="https://github.com/kevinalexandr19/manual-python-geologia" target="_blank">
            <img src="https://img.shields.io/github/stars/kevinalexandr19/manual-python-geologia.svg?style=social&label=Github Repo">
        </a>
        &nbsp;&nbsp;
        <!-- Licencia -->
        <img src="https://img.shields.io/github/license/kevinalexandr19/manual-python-geologia.svg?color=forestgreen">
        &nbsp;&nbsp;
        <!-- Release date -->
        <img src="https://img.shields.io/github/release-date/kevinalexandr19/manual-python-geologia?color=gold">
    </span>
    <br>
    <span>
        <!-- Perfil de LinkedIn -->
        <a target="_blank" href="https://www.linkedin.com/in/kevin-alexander-gomez/">
            <img src="https://img.shields.io/badge/-Kevin Alexander Gomez-5eba00?style=social&logo=linkedin">
        </a>
        &nbsp;&nbsp;
        <!-- Perfil de Github -->
        <a target="_blank" href="https://github.com/kevinalexandr19">
            <img src="https://img.shields.io/github/followers/kevinalexandr19.svg?style=social&label=kevinalexandr19&maxAge=2592000">
        </a>
    </span>
    <br>
</div>

***

<span style="color:lightgreen; font-size:25px">**PG303 - Análisis geoespacial** </span>

Bienvenido al curso!!!

Vamos a revisar algunas aplicaciones del <span style="color:gold">análisis geoespacial</span> usando código en Python. <br>
Es necesario que tengas un conocimiento previo en programación con Python, estadística y sistemas de información geográfica.


<span style="color:gold; font-size:20px">**Leafmap y Folium**</span>

***
- [Capas de un mapa interactivo](#parte-1)
- [Visualizar un ráster](#parte-2)
- [Agregar países desde un shapefile](#parte-3)
- [Comprobar la alineación](#parte-4)

***

<a id="parte-1"></a>

### <span style="color:lightgreen">**Capas de un mapa interactivo**</span>
***

<span style="color:gold">Leafmap</span> reúne herramientas para explorar datos geoespaciales en Jupyter. Uno de sus motores de visualización es <span style="color:gold">Folium</span>, que genera un mapa HTML con Leaflet.

El ejemplo original servía el GeoTIFF mediante `Leafmap.add_raster` y un servidor local de teselas. En JupyterLite usamos Folium directamente. El kernel prepara una imagen de los valores de elevación y la incluye en el HTML con sus límites. Los polígonos del shapefile se incluyen como GeoJSON.

| Capa | Datos que recibe | Trabajo que hace el código |
| --- | --- | --- |
| Elevación | GeoTIFF del cuaderno de Rioxarray | Reproyectar, colorear y ubicar la imagen |
| Límites | Shapefile del cuaderno de GeoPandas | Leer geometrías y convertirlas a GeoJSON |
| Mapa base opcional | Teselas de OpenStreetMap | Pedir al servidor las teselas de la vista |

La vista interactiva carga Leaflet desde internet. Los datos de los ejemplos se incluyen con el curso. En Python local puedes instalar `python -m pip install geopandas fiona matplotlib "rioxarray==0.18.2" "folium==0.18.0"`.

In [ ]:
import sys

if sys.platform == "emscripten":
    import piplite
    from pyodide_js import loadPackage

    await loadPackage(["rasterio", "xarray", "geopandas", "fiona", "matplotlib",
                       "requests", "ssl", "jinja2", "xyzservices"])
    await piplite.install(["rioxarray==0.18.2", "folium==0.18.0", "branca==0.8.1"])

<a id="parte-2"></a>

### <span style="color:lightgreen">**Visualizar un ráster**</span>
***

El archivo original tiene coordenadas en EPSG:4326 (grados). La vista de Leaflet usa **Web Mercator** (EPSG:3857). Reproyectamos las celdas a esa malla con interpolación bilineal, adecuada para una visualización de elevación continua. La imagen mantiene la fila norte arriba.

Folium recibe los límites de la imagen como pares `[latitud, longitud]` en grados. Por ello transformamos **los límites** de la nueva malla de vuelta a EPSG:4326. Compara ese orden con `(x, y)` de Shapely y con `[longitud, latitud]` de GeoJSON.

In [ ]:
from pathlib import Path

import folium
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import rioxarray as rxr
from rasterio.enums import Resampling
from rasterio.io import MemoryFile
from rasterio.warp import transform_bounds

ruta = Path("files/peru_elevacion_60s.tif")
with MemoryFile(ruta.read_bytes()) as archivo, archivo.open() as fuente:
    raster = rxr.open_rasterio(fuente, masked=True)
    dem = raster.isel(band=0, drop=True).load()

web = dem.rio.reproject("EPSG:3857", resampling=Resampling.bilinear)
oeste, sur, este, norte = transform_bounds(web.rio.crs, "EPSG:4326", *web.rio.bounds())
limites = [[sur, oeste], [norte, este]]
print(f"Original: {dem.rio.crs} {dem.shape} | Imagen web: {web.rio.crs} {web.shape}")
print("Límites [latitud, longitud]:", np.round(limites, 4))

Elegimos una escala con el nivel del mar en 0 m. `rgba` guarda cuatro canales por píxel, rojo, verde, azul y opacidad. Los píxeles sin datos quedan transparentes. La figura estática usa esos mismos colores, con coordenadas de la malla web en metros.

In [ ]:
elevacion = web.values
validos = elevacion[np.isfinite(elevacion)]
escala = mpl.colors.TwoSlopeNorm(vmin=float(validos.min()), vcenter=0, vmax=float(validos.max()))
paleta = mpl.colormaps["terrain"]
rgba = paleta(escala(np.ma.masked_invalid(elevacion)), bytes=True)

x0, y0, x1, y1 = web.rio.bounds()
fig, ax = plt.subplots(figsize=(6.7, 4.5), layout="constrained")
ax.imshow(rgba, extent=(x0, x1, y0, y1), origin="upper")
barra = mpl.cm.ScalarMappable(norm=escala, cmap=paleta)
fig.colorbar(barra, ax=ax, label="Elevación (m)", shrink=.8)
ax.set(title="Imagen que se superpone en el mapa", xlabel="X · Web Mercator (m)",
       ylabel="Y · Web Mercator (m)")
ax.ticklabel_format(style="plain", useOffset=False)
plt.show()

<a id="parte-3"></a>

### <span style="color:lightgreen">**Agregar países desde un shapefile**</span>
***

GeoPandas lee los límites del mismo archivo que usamos en el cuaderno anterior. Seleccionamos Sudamérica y conservamos solo el nombre y la geometría para la capa. `to_crs("EPSG:4326")` prepara las coordenadas que espera GeoJSON. El estilo define el color y el grosor de las líneas. La transparencia del relleno deja ver la elevación debajo.

In [ ]:
paises = gpd.read_file("files/countries/countries.shp", engine="fiona")
region = paises.loc[paises["continent"] == "South America", ["name", "geometry"]].to_crs("EPSG:4326")
print(f"{len(region)} geometrías vectoriales · {region.crs}")
region.head()

<a id="parte-4"></a>

### <span style="color:lightgreen">**Comprobar la alineación**</span>
***

Cada capa tiene un nombre y un interruptor. `fit_bounds` centra la vista sobre el DEM. Agregamos como referencia la misma celda del cuaderno de Rioxarray, ubicada con su latitud y longitud. Pasa el cursor sobre un país para consultar su nombre. Apaga y enciende la imagen y los límites por separado para examinar cómo encajan.

In [ ]:
mapa = folium.Map(tiles=None, height=470, control_scale=True)
folium.TileLayer("OpenStreetMap", name="Mapa base · OpenStreetMap", show=False).add_to(mapa)
folium.raster_layers.ImageOverlay(
    rgba, bounds=limites, name="Elevación · Perú central", opacity=.85, origin="upper",
).add_to(mapa)
folium.GeoJson(
    region, name="Límites de los países",
    style_function=lambda _: {"color": "#213947", "weight": 2, "fillOpacity": 0},
    tooltip=folium.GeoJsonTooltip(fields=["name"], aliases=["País"]),
).add_to(mapa)

celda = dem.isel(y=100, x=150)
folium.CircleMarker(
    [float(celda.y), float(celda.x)], radius=6, color="#152B39",
    fill=True, fill_opacity=1, fill_color="#F4CE61",
    tooltip=f"Celda del cuaderno de Rioxarray · {float(celda):.1f} m",
).add_to(mapa)
folium.LayerControl(collapsed=False).add_to(mapa)
mapa.fit_bounds(limites)
mapa

**Comprueba tu lectura.** ¿Qué capas necesitan teselas de un servicio web y cuáles vienen de los archivos? Identifica en el código dónde se cambia el CRS de la malla y dónde se cambia el orden de las coordenadas. Revisa el marcador sobre el ráster con la capa de límites encendida. Un mapa puede verse convincente aunque los datos estén desplazados.

[Leafmap y su motor de Folium](https://leafmap.org/foliumap/) · [Capas ráster de Folium](https://python-visualization.github.io/folium/latest/reference.html#folium.raster_layers.ImageOverlay)